# Notebook 2 — Regression


## Learning objectives

- Define regression and write down the linear regression model in vector form.
- State the mean-squared-error loss and the closed-form (normal-equations) solution.
- Compute and interpret MAE, RMSE, MAPE, and $R^{2}$ on a real test set.
- Compare a chronological time-based split with a random split, and observe leakage.
- Inspect coefficients, recognise multicollinearity, and apply ridge regression as a remedy.

## 2.1 Definition

**Regression** is supervised learning whose target $y$ is a continuous real number. Examples in atmospheric science:

- predicting $\text{PM}_{2.5}$ in $\mu\text{g}/\text{m}^{3}$;
- predicting tomorrow's daily mean temperature in °C;
- predicting boundary-layer height in metres.

A regression model is any function $\hat{f}: \mathbb{R}^{d} \to \mathbb{R}$ that maps a feature vector to a real number. The simplest such function is **linear regression**.

## 2.2 The linear regression model

> **Definition.** Let $\mathbf{x} \in \mathbb{R}^{d}$ be a feature vector. *Linear regression* models the conditional mean of $y$ as
>
> $$\hat{y} = \hat{f}(\mathbf{x}) = w_{0} + w_{1} x_{1} + w_{2} x_{2} + \dots + w_{d} x_{d} = w_{0} + \mathbf{w}^{\top} \mathbf{x},$$
>
> where $w_{0} \in \mathbb{R}$ is the **intercept** (or bias) and $\mathbf{w} = (w_{1},\dots,w_{d})^{\top} \in \mathbb{R}^{d}$ is the **coefficient vector**.

The model has $d+1$ scalar parameters. Each $w_{j}$ is interpreted as the change in the predicted $\hat{y}$ for a unit increase in feature $x_{j}$, holding all other features fixed.

In this notebook, with `top_n=4` correlated neighbour stations, we will have $d = 8$ features (PM2.5 + PM10 from each of four stations), and the model takes the form

$$\widehat{\text{PM}_{2.5}}_{\text{ATZX}} = w_{0} + w_{1}\,\text{pm25}_{\text{GY}} + \dots + w_{8}\,\text{pm10}_{\text{WL}}.$$

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path('data')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (9, 5)

df = pd.read_csv(DATA_DIR / 'air_quality_basic_aotizhongxin.csv', parse_dates=['datetime'])
df = df.sort_values('datetime').reset_index(drop=True)
print('Rows:', len(df))

## 2.3 Loss function: mean squared error

We need a numeric definition of "close". The standard choice in regression is the **squared loss**

$$L(\hat{y}, y) = (\hat{y} - y)^{2}.$$

Averaged over the training set, this becomes the **mean squared error (MSE)**:

$$\text{MSE}(\mathbf{w}, w_{0}) = \frac{1}{N} \sum_{i=1}^{N} \bigl( y_{i} - w_{0} - \mathbf{w}^{\top}\mathbf{x}_{i} \bigr)^{2}.$$

We square (rather than take absolute value) for two reasons: (i) the function is differentiable everywhere, enabling closed-form solutions and easy gradient computation; (ii) squared loss penalises large errors disproportionately, which matches our intuition that being off by 100 μg/m³ is more than ten times worse than being off by 10 μg/m³.

## 2.4 Geometric interpretation

In the special case $d = 1$ (one feature), the regression problem reduces to fitting a straight line $\hat{y} = w_{0} + w_{1}x$ through a scatter of points $(x_{i}, y_{i})$. MSE is the average squared *vertical* distance from each point to the line. The optimal line is the one that minimises this average.

In $d$ dimensions the line generalises to a $d$-dimensional **hyperplane**.

## 2.5 Closed-form solution (the normal equations)

Stack the feature vectors into a matrix $\mathbf{X} \in \mathbb{R}^{N \times (d+1)}$ where the first column is all ones (for the intercept). Stack the labels into $\mathbf{y} \in \mathbb{R}^{N}$, and the parameters into $\boldsymbol{\beta} = (w_{0}, w_{1}, \dots, w_{d})^{\top}$. Setting the gradient to zero gives the **normal equations**:

$$\boldsymbol{\hat{\beta}} = (\mathbf{X}^{\top} \mathbf{X})^{-1} \mathbf{X}^{\top} \mathbf{y}.$$

Provided the columns of $\mathbf{X}$ are linearly independent (no perfectly redundant features), this closed-form solution is unique. `scikit-learn`'s `LinearRegression` computes it for us; we never write the matrix inversion by hand.

When two features are highly correlated (e.g., $\text{PM}_{2.5}$ at *Dongsi* and *Nongzhanguan* have an empirical Pearson correlation of $0.964$ in this dataset), the columns are *nearly* linearly dependent, $\mathbf{X}^{\top}\mathbf{X}$ becomes ill-conditioned, and $\boldsymbol{\hat{\beta}}$ becomes unstable. The remedy — **ridge regression** — appears in §2.9.

## 2.6 Worked example — pick the four most-correlated neighbour stations

We rank neighbour stations by Pearson correlation between their PM2.5 and the target's $\text{PM}_{2.5}$ (computed *on the training set only*, never the full data — that would leak).

In [ ]:
# Train period = first 80% (chronological)
cut = int(0.8 * len(df))
train_df, test_df = df.iloc[:cut].copy(), df.iloc[cut:].copy()

station_cols = [c for c in df.columns if c.startswith('pm25_')]
target_col = 'target_pm25'

corr = train_df[[target_col] + station_cols].corr()[target_col].drop(target_col)
ranked = corr.sort_values(ascending=False)
print('Top-5 correlated stations (training set only):')
print(ranked.head())

top4_stations = [c.replace('pm25_', '') for c in ranked.head(4).index]
print('\nSelected:', top4_stations)

In [ ]:
feature_cols = []
for st in top4_stations:
    feature_cols += [f'pm25_{st}', f'pm10_{st}']
print('Features:', feature_cols)

for part in (train_df, test_df):
    part.dropna(subset=feature_cols + [target_col], inplace=True)

x_train, y_train = train_df[feature_cols], train_df[target_col]
x_test, y_test = test_df[feature_cols], test_df[target_col]
print(f'Train: {len(x_train):,}, Test: {len(x_test):,}')

### Fit the model

We wrap the scaler and regressor in a `Pipeline` so that scaling is fitted on `x_train` only — the rule from Notebook 1.

In [ ]:
pipe = Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())])
pipe.fit(x_train, y_train)
pred = pipe.predict(x_test)
print('Model fitted on', len(x_train), 'rows.')

## 2.7 Evaluating regression algorithms

A trained regression model produces predictions $\{\hat{y}_{i}\}_{i=1}^{N_{\text{test}}}$ on the test set. We summarise their quality with four standard metrics.

### 2.7.1 Mean Absolute Error (MAE)

$$\text{MAE} = \frac{1}{N_{\text{test}}} \sum_{i=1}^{N_{\text{test}}} \lvert y_{i} - \hat{y}_{i} \rvert.$$

Same units as $y$ ($\mu\text{g}/\text{m}^{3}$ for $\text{PM}_{2.5}$). Robust to outliers.

### 2.7.2 Root Mean Squared Error (RMSE)

$$\text{RMSE} = \sqrt{\frac{1}{N_{\text{test}}} \sum_{i=1}^{N_{\text{test}}} (y_{i} - \hat{y}_{i})^{2}}.$$

Same units as $y$; penalises large errors more heavily than MAE.

### 2.7.3 Mean Absolute Percentage Error (MAPE)

$$\text{MAPE} = \frac{1}{N_{\text{test}}} \sum_{i=1}^{N_{\text{test}}} \left| \frac{y_{i} - \hat{y}_{i}}{y_{i}} \right|.$$

Unitless. Useful when comparing models across pollutants. **Caveat**: undefined when any $y_{i} = 0$, and explodes when $y_{i}$ is small.

### 2.7.4 Coefficient of determination $R^{2}$

$$R^{2} = 1 - \frac{\sum_{i} (y_{i} - \hat{y}_{i})^{2}}{\sum_{i} (y_{i} - \bar{y})^{2}}.$$

Fraction of test-set variance explained. $R^{2} = 1$ is perfect; $R^{2} = 0$ means the model is no better than always predicting the mean; $R^{2} < 0$ means the model is *worse* than always predicting the mean.

In [ ]:
def report(y_true, y_pred, label='model'):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mape = float(np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, np.nan, y_true))))
    r2   = r2_score(y_true, y_pred)
    print(f'{label:>30s}  MAE={mae:6.2f}   RMSE={rmse:6.2f}   MAPE={mape:6.2%}   R2={r2:6.3f}')

report(y_test.values, pred, label='LinearRegression(top_n=4)')

### 2.8 Diagnostic plots

Two plots are essential after fitting any regression: (a) **actual vs predicted** scatter and (b) the **test-period overlay** showing predicted and actual time series side-by-side.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
# Actual vs predicted scatter
axes[0].scatter(y_test, pred, alpha=0.2, s=10)
lo, hi = float(min(y_test.min(), pred.min())), float(max(y_test.max(), pred.max()))
axes[0].plot([lo, hi], [lo, hi], 'r--')
axes[0].set_xlabel('Actual PM2.5'); axes[0].set_ylabel('Predicted PM2.5')
axes[0].set_title('Actual vs Predicted')
# Test-period overlay (first 500 rows)
plot_df = pd.DataFrame({'datetime': test_df['datetime'].values[:500], 'actual': y_test.values[:500], 'pred': pred[:500]})
axes[1].plot(plot_df['datetime'], plot_df['actual'], label='Actual', lw=1.5)
axes[1].plot(plot_df['datetime'], plot_df['pred'], label='Predicted', lw=1.5, alpha=0.8)
axes[1].set_title('Test-period overlay (first 500 hours)'); axes[1].legend()
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30)
plt.tight_layout(); plt.show()

Points clustered close to the red 1:1 line on the left and well-tracked actuals on the right indicate a good fit. The clusters far from the diagonal at high $\text{PM}_{2.5}$ are the *pollution episodes* the model under-predicts — a familiar failure mode that more capable models in later notebooks will partially fix.

## 2.9 Time split vs random split — the leakage demo

We deliberately train the same model under a *random* split (which is wrong for time-series) and compare the resulting metrics to the chronological split. Random splits look artificially better because future hours leak into training.

In [ ]:
# Random split (deliberately wrong for time series)
rand_train, rand_test = train_test_split(df.dropna(subset=feature_cols + [target_col]),
                                         test_size=0.2, random_state=42, shuffle=True)
pipe_rand = Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())])
pipe_rand.fit(rand_train[feature_cols], rand_train[target_col])
pred_rand = pipe_rand.predict(rand_test[feature_cols])
report(rand_test[target_col].values, pred_rand, label='LinearRegression (random split)')
report(y_test.values, pred,                 label='LinearRegression (time split)')

Random splits *look* better because future hours leak into training. Time splits are honest. **For all subsequent notebooks we use chronological splits exclusively.**

## 2.10 Coefficients and multicollinearity

The four $\text{PM}_{2.5}$ features are highly correlated. Their individual coefficients are unstable, even though the joint prediction is accurate. Below we show both the bar plot of standardised coefficients and the pairwise correlation heatmap that explains why the magnitudes are not directly interpretable.

In [ ]:
scaler = pipe.named_steps['scaler']
lr = pipe.named_steps['model']
coefs = pd.Series(lr.coef_, index=feature_cols).sort_values()
fig, ax = plt.subplots(figsize=(7, 5))
coefs.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Standardised regression coefficients')
plt.tight_layout(); plt.show()

# Pairwise correlations between the four PM2.5 features
sns.heatmap(x_train[[c for c in feature_cols if c.startswith('pm25_')]].corr(),
            annot=True, fmt='.2f', cmap='Blues')
plt.title('PM2.5 feature correlations')
plt.tight_layout(); plt.show()

Because the four $\text{PM}_{2.5}$ columns correlate above 0.94 with each other, the regression *can* split the credit between them in many essentially equivalent ways. This is **multicollinearity**. The joint hyperplane is well-determined; the individual weights are not.

## 2.11 Ridge regression — a brief preview

When features are correlated, ordinary least squares produces unstable coefficients. **Ridge regression** adds a quadratic penalty:

$$\hat{\boldsymbol{\beta}}_{\text{ridge}} = \arg\min_{\boldsymbol{\beta}} \; \lVert \mathbf{y} - \mathbf{X}\boldsymbol{\beta} \rVert_{2}^{2} + \alpha \lVert \boldsymbol{\beta} \rVert_{2}^{2},$$

where $\alpha > 0$ controls how strongly we shrink coefficients toward zero. The closed-form solution becomes

$$\hat{\boldsymbol{\beta}}_{\text{ridge}} = (\mathbf{X}^{\top}\mathbf{X} + \alpha \mathbf{I})^{-1} \mathbf{X}^{\top}\mathbf{y},$$

which is always invertible for $\alpha > 0$ even when $\mathbf{X}^{\top}\mathbf{X}$ is singular.

In [ ]:
for alpha in [0.1, 1.0, 10.0, 100.0]:
    p = Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=alpha))])
    p.fit(x_train, y_train)
    report(y_test.values, p.predict(x_test), label=f'Ridge alpha={alpha}')

Small $\alpha$ barely changes performance; large $\alpha$ shrinks all coefficients toward zero, eventually under-fitting. The sweet spot is usually picked by cross-validation (we automate this in Notebook 6 with `TimeSeriesSplit` + `GridSearchCV`).

## 2.12 Common misconceptions

**"A high $R^{2}$ means the model is good."** $R^{2}$ measures variance *explained*, not error in absolute units. An $R^{2}$ of 0.94 sounds excellent, but if the test-set standard deviation of $\text{PM}_{2.5}$ is $90~\mu\text{g}/\text{m}^{3}$, an RMSE of $22$ still translates to forecasts that miss real values by $20$+ $\mu\text{g}/\text{m}^{3}$ on average. Always report MAE / RMSE alongside $R^{2}$.  

**"Negative $R^{2}$ is impossible."** It is not. A linear regression that uses only current-hour meteorology to predict next-hour $\text{SO}_{2}$ can yield a *negative* $R^{2}$ on this dataset, indicating that those features actively hurt rather than help.  

**"Linear regression assumes the residuals are normal."** Only the t-tests and confidence intervals derived from linear regression assume Gaussian residuals. The point estimate $\hat{\boldsymbol{\beta}}$ is the *minimum-MSE* solution under no distributional assumption.  

**"You should not standardise before linear regression."** For ordinary least squares, scaling does not change the predictions, but it does change the *magnitudes* of the coefficients and the conditioning of $\mathbf{X}^{\top}\mathbf{X}$. For ridge, lasso, gradient-based fitters, KNN, SVMs, and neural networks, scaling is essential.  

**"Correlation = causation."** Pearson's $r = 0.96$ between $\text{PM}_{2.5}$ at *Aotizhongxin* and *Guanyuan* does not mean *Guanyuan* causes *Aotizhongxin*'s pollution. Both stations sit in the same urban airshed.

---
## Exercises

### Exercise 2.1 — `top_n` sweep

*Hint: rerun §2.6 inside a loop over `n in [1, 2, 4, 8, 11]`. Report MAE for each. Where does the curve flatten? At what $n$ would you stop adding stations?*

In [ ]:
# EXERCISE 2.1
# results = {}
# for n in [1, 2, 4, 8, 11]:
#     stations = [c.replace('pm25_', '') for c in ranked.head(n).index]
#     feats = [f'pm25_{s}' for s in stations] + [f'pm10_{s}' for s in stations]
#     ...
# TODO: plot MAE vs n.


### Exercise 2.2 — interpret residuals

*Hint: residuals = `y_test - pred`. Plot a histogram and a residual-vs-time line chart. Are residuals centred on zero? Any visible structure (e.g., a winter bias)?*

In [ ]:
# EXERCISE 2.2
# residuals = y_test.values - pred
# TODO: histogram + time series of residuals, with axhline at 0.


## 2.12 Chapter summary

- A regression model maps $\mathbb{R}^{d} \to \mathbb{R}$; linear regression chooses $\hat{y} = w_{0} + \mathbf{w}^{\top}\mathbf{x}$ and fits $w$ by minimising mean squared error.
- The closed-form solution $\boldsymbol{\hat{\beta}} = (\mathbf{X}^{\top}\mathbf{X})^{-1}\mathbf{X}^{\top}\mathbf{y}$ is unique when features are not collinear; ridge regression adds $\alpha\mathbf{I}$ to stabilise the inversion when they are.
- Always report MAE, RMSE, MAPE *and* $R^{2}$ together — each metric tells a different story.
- The basic regression at *Aotizhongxin* with four correlated neighbour stations achieves $R^{2}\approx 0.94$, and `baseline_plus_lags` reaches the same level at $h=1$ on the advanced forecasting dataset.
- Multicollinearity makes individual coefficient values unreliable for *interpretation* even when the joint prediction is accurate; never read a single weight as a causal effect.

**Next:** Notebook 3 swaps the continuous target for a categorical one and introduces logistic regression.